# Trace-task learning

Trace tasks keep difficulty fixed within a run, so validation loss and task-specific generation quality can be interpreted as ordinary learning curves. This notebook intentionally excludes BBH curricula.

In [ ]:
from pathlib import Path
from statistics import median
import sys

import matplotlib
if "ipykernel" in sys.modules:
    matplotlib.use("module://matplotlib_inline.backend_inline")
import matplotlib.pyplot as plt

def find_repo_root(start):
    for candidate in (start, *start.parents):
        if (candidate / "experiments").is_dir() and (candidate / "figures" / "plotting_utils.py").is_file():
            return candidate
    raise FileNotFoundError(f"Could not locate repository root above {start}")

REPO_ROOT = find_repo_root(Path.cwd().resolve())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from figures.plotting_utils import (
    ARCHITECTURE_COLORS,
    filter_records,
    grouped,
    load_training_records,
    metric_label,
    plot_seed_and_median_curves,
    primary_metric,
    set_plot_style,
    unique_values,
)

set_plot_style()
RESULT_ROOT = REPO_ROOT / "results" / "trace"
FIGURE_DIR = REPO_ROOT / "figures"


In [ ]:
records = load_training_records(RESULT_ROOT) if RESULT_ROOT.exists() else []
records = [row for row in records if row.get("level") is None]
print(f"Loaded {len(records)} trace evaluation checkpoints from {RESULT_ROOT}")
print("tasks:", unique_values(records, "task"))
print("architectures:", unique_values(records, "architecture"))
print("devices:", unique_values(records, "device"))
print("seeds:", unique_values(records, "seed"))
if not records:
    print("No trace training artifacts are present yet; run a trace preset before plotting.")


## Select one fixed-difficulty task

Loss and generation quality are shown together because their difficulty does not change during training. The third panel uses a harder task-specific slice of the same success criterion where available.

In [ ]:
TASK = "othello"  # shortest_path, othello
DEVICE = None
SHORTEST_PATH_DISTRIBUTION = "main"
ARCHITECTURES = list(ARCHITECTURE_COLORS)
SUCCESS_METRIC = primary_metric(TASK)
SECONDARY_METRICS = {
    "shortest_path": "optimal_path_long",
    "othello": "sequence_legality",
}
SECONDARY_METRIC = SECONDARY_METRICS[TASK]

selected = filter_records(records, task=TASK, device=DEVICE)
if TASK == "shortest_path":
    selected = filter_records(selected, shortest_path_distribution=SHORTEST_PATH_DISTRIBUTION)
selected = [row for row in selected if row.get("architecture") in ARCHITECTURES]
print(f"Selected {len(selected)} checkpoints from {len({row['run_dir'] for row in selected})} runs")


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4.3))
if not selected:
    for ax in axes:
        ax.set_axis_off()
    fig.text(
        0.5, 0.5,
        f"No {TASK.replace('_', ' ')} trace artifacts found.\n"
        "Run its trace preset, then rerun this notebook.",
        ha="center", va="center", fontsize=12,
    )
else:
    plot_seed_and_median_curves(axes[0], selected, metric="loss")
    axes[0].set_title("Next Legal Moves Loss (teacher forced)")
    axes[0].set_yscale("log")
    plot_seed_and_median_curves(axes[1], selected, metric=SUCCESS_METRIC)
    axes[1].set_title(metric_label(SUCCESS_METRIC))
    axes[1].set_ylim(-0.02, 1.02)
    plot_seed_and_median_curves(axes[2], selected, metric=SECONDARY_METRIC)
    axes[2].set_title(metric_label(SECONDARY_METRIC))
    axes[2].set_ylim(-0.02, 1.02)
    handles, labels = axes[0].get_legend_handles_labels()
    for ax in axes:
        legend = ax.get_legend()
        if legend:
            legend.remove()
    if handles:
        fig.legend(handles, labels, loc="lower center", ncol=len(labels), bbox_to_anchor=(0.5, -0.04))
fig.suptitle(TASK.replace("_", " ").title(), y=1.03)
fig.tight_layout()
plt.show()
fig.savefig(FIGURE_DIR / "trace_plot_figs.png", dpi=220, bbox_inches="tight")


## Shortest-path accuracy by generation step

Step 1 is the first transition after the supplied start node. Later steps include only examples whose target paths are long enough to contain that transition.

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 4.4))
if TASK != "shortest_path" or not selected:
    ax.text(0.5, 0.5, "Select shortest_path runs to show step accuracy", ha="center", va="center")
    ax.set_axis_off()
else:
    final_rows = []
    for (_run_dir,), rows in grouped(selected, "run_dir").items():
        final_rows.append(max(rows, key=lambda row: row["step"]))
    for architecture in ARCHITECTURES:
        rows = [row for row in final_rows if row.get("architecture") == architecture]
        if not rows:
            continue
        steps = sorted({
            int(key.removeprefix("path_step_").removesuffix("_accuracy"))
            for row in rows for key in row
            if key.startswith("path_step_") and key.endswith("_accuracy")
        })
        values = [median([row[f"path_step_{step}_accuracy"] for row in rows if f"path_step_{step}_accuracy" in row]) for step in steps]
        ax.plot(steps, values, marker="o", label=architecture,
                color=ARCHITECTURE_COLORS.get(architecture))
    ax.set_xlabel("Shortest-path transition step")
    ax.set_ylabel("Free-generation token accuracy")
    ax.set_ylim(-0.02, 1.02)
    ax.set_xticks(range(1, 11))
    ax.set_title("Final checkpoint: accuracy by path step")
    ax.legend()
fig.tight_layout()
plt.show()
# fig.savefig(FIGURE_DIR / "shortest_path_step_accuracy.png", dpi=220, bbox_inches="tight")


## Measured training throughput

Throughput is kept separate from quality and should only be compared within one task, device, batch size, and sequence format.

In [ ]:
final_by_run = []
for (_run_dir,), rows in grouped(selected, "run_dir").items():
    rows = sorted(rows, key=lambda row: row["step"])
    if rows and rows[-1].get("train_tok_per_s") is not None:
        final_by_run.append(rows[-1])

fig, ax = plt.subplots(figsize=(8.5, 4.4))
if not final_by_run:
    ax.text(0.5, 0.5, "No selected throughput records", ha="center", va="center")
    ax.set_axis_off()
else:
    for index, architecture in enumerate(ARCHITECTURES):
        values = [row["train_tok_per_s"] for row in final_by_run if row.get("architecture") == architecture]
        if not values:
            continue
        offsets = [(item - (len(values) - 1) / 2) * 0.07 for item in range(len(values))]
        ax.scatter([index + offset for offset in offsets], values,
                   color=ARCHITECTURE_COLORS.get(architecture), alpha=0.65)
        ax.hlines(median(values), index - 0.28, index + 0.28, color="black", linewidth=2)
    ax.set_xticks(range(len(ARCHITECTURES)), [name.replace("_", "\n") for name in ARCHITECTURES])
    ax.set_ylabel("Training tokens / second")
    ax.set_title(f"{TASK}: final measured throughput ({DEVICE or 'mixed devices'})")
fig.tight_layout()
plt.show()
# fig.savefig(FIGURE_DIR / f"{TASK}_trace_throughput.png", dpi=220, bbox_inches="tight")
